In [ ]:
from flax import nnx
from flax.typing import Array, Shape
from jax import numpy as jnp
from flax.nnx.module import first_from
import jax

def softmax_cross_attention(q: jnp.ndarray, k: jnp.ndarray, v: jnp.ndarray, *, mask: None | jnp.ndarray, is_causal: bool):
    T_q, T_k = q.shape[-3], k.shape[-3]
    d_head = q.shape[-1]

    # assumes on decode we only have 1 token
    causal_mask = jnp.tri(T_q, T_k, k=T_k-T_q, dtype=bool) if is_causal else jnp.ones((T_q, T_k), dtype=bool)

    if mask is not None:
        causal_mask = causal_mask & mask

    attn_scores: Array = jnp.einsum("...qhd,...khd->...hqk", q, k) / (d_head ** 0.5)
    attn_scores = jnp.where(causal_mask, attn_scores, -jnp.inf)

    attn_weights = nnx.softmax(attn_scores, axis=-1)

    return jnp.einsum("...hqk,...khd->...qhd", attn_weights, v)

class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, d_head: int, *, decode = False, rngs: nnx.Rngs):
        assert d_model % d_head == 0
        n_head = d_model // d_head

        projection = lambda : nnx.Einsum(
            "...e,ehd->...hd",
            (d_model, n_head, d_head),
            (n_head, d_head),
            rngs=rngs
        )

        self.W_q = projection()
        self.W_k = projection()
        self.W_v = projection()

        self.W_o = nnx.Einsum(
            "...hd,hde->...e",
            (n_head, d_head, d_model),
            (d_model,),
            rngs=rngs
        )

        self.decode = decode

        self.cached_k: None | nnx.Cache[Array] = nnx.data(None)
        self.cached_v: None | nnx.Cache[Array] = nnx.data(None)
        self.cached_index: None | nnx.Cache[Array] = nnx.data(None)

    def init_cache(self, shape: Shape):
        self.cached_k = nnx.Cache(jnp.zeros(shape)) # bhkd
        self.cached_v = nnx.Cache(jnp.zeros(shape))
        self.cached_index = nnx.Cache(jnp.array(0))

    def __call__(self, X: jnp.ndarray, memory: jnp.ndarray | None, *, decode: None | bool = None, is_causal = False):
        if memory is None:
            memory = X

        q = self.W_q(X)
        k = self.W_k(memory)
        v = self.W_v(memory)

        mask = None

        if first_from(decode, self.decode, error_msg="no decode set"):
            assert self.cached_k is not None
            assert self.cached_v is not None
            assert self.cached_index is not None

            (*batch_dims, max_length, num_kv_heads, depth_per_head) = self.cached_k.shape

            cur_index = self.cached_index[...]
            zero = jnp.array(0)
            indices = (zero,) * len(batch_dims) + (cur_index, zero, zero)
            # .at.set also possible to use here
            k = jax.lax.dynamic_update_slice(self.cached_k[...], k, indices)
            v = jax.lax.dynamic_update_slice(self.cached_v[...], v, indices)
            self.cached_k[...] = k
            self.cached_v[...] = v
            self.cached_index[...] += 1 # assumes on decode we only have 1 token

            mask = jnp.broadcast_to( # assumes on decode we only have 1 token
                jnp.arange(max_length) <= cur_index,
                tuple(batch_dims) + (1, 1, max_length),
            )

        y = softmax_cross_attention(q, k, v, is_causal=is_causal, mask=mask)
        return self.W_o(y)        

In [ ]:
import jax
import jax.numpy as jnp

@jax.jit
def index(x, u, i):
    return x.at[i].set(u[0])

@jax.jit
def slice_(x, u, i):
    return x.at[i:i+1].set(u)

@jax.jit
def dynamic_slice(x, u, i):
    return jax.lax.dynamic_update_slice(x, u, (i,))

x = jnp.zeros(5)
u = jnp.array([9.])

for i in range(3):
    for fn in (index, slice_, dynamic_slice):
        try:
            print(fn.__name__, fn(x, u, i))
        except IndexError:
            print(fn.__name__, "FAILED")

index [9. 0. 0. 0. 0.]
slice_ FAILED
dynamic_slice [9. 0. 0. 0. 0.]
index [0. 9. 0. 0. 0.]
slice_ FAILED
dynamic_slice [0. 9. 0. 0. 0.]
index [0. 0. 9. 0. 0.]
slice_ FAILED
dynamic_slice [0. 0. 9. 0. 0.]


In [ ]:
# JIT compatible: dynamic integer indexing
def index(x, u, i):
    return x.at[i].set(u[0])

# JIT incompatible when i is dynamic
# Works without JIT, or when i is static
def slice_(x, u, i):
    return x.at[i:i+1].set(u)

# JIT compatible: dynamic slice position
def dynamic_slice(x, u, i):
    return jax.lax.dynamic_update_slice(x, u, (i,))


In [9]:
from flax.nnx import combine_masks

batch_dims = (3,)
cur_index = 5
max_length = 10

mask = combine_masks(
        None,
        jnp.broadcast_to(
          jnp.arange(max_length) <= cur_index,
          tuple(batch_dims) + (3, 1, max_length),
        ),
      )

mask, (jnp.arange(max_length) <= cur_index).shape

(Array([[[[1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]],
 
         [[1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]],
 
         [[1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]]],
 
 
        [[[1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]],
 
         [[1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]],
 
         [[1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]]],
 
 
        [[[1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]],
 
         [[1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]],
 
         [[1., 1., 1., 1., 1., 1., 0., 0., 0., 0.]]]], dtype=float32),
 (10,))

In [2]:
(0,) * 5 + (1,2,3) + (3,4)

(0, 0, 0, 0, 0, 1, 2, 3, 3, 4)